In [ ]:
import operator
import os
import re
from groq import Groq
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field

In [ ]:
class FraudState(TypedDict):
    job_text:str
    suspicious_keyword:Annotated[list[str],operator.add]
    risk_score:int
    tool_call_iteration:int
    emailaddress:list[str]
    is_email_free:bool
    website_link:list[str]
    website_available:bool
    Linkedin_profile:bool
    evidance_log:Annotated[list[str],operator.add]



In [ ]:
class SuspiciousPhraseExtractor(BaseModel):
    """Schema to force the llm to output a clean list of suspicious phrases"""
    phrases:list[str]=Field(description="List of exact phrases extracted from the text that indicate upfront payments, unrealistic salary promises, or pressure tactics."
    )


In [ ]:
llm=Groq.LLM(api_key=os.environ.get("Groq_API_KEY"),model="qwen/qwen3.8-27b",temperature=0,max_tokens=512)
structured_llm=llm.structured_output(SuspiciousPhraseExtractor)

In [ ]:
@tool
def search_tool(company_name:str,website_link:str)->TavilySearchResults:
    """Searches Tavily to verify if a hiring company has a real website 
    or registered online footprint in Bangladesh.
    
    Args:
        company_name: Name of the hiring company extracted from the job circular.
        website_link: Optional website link found in the post (e.g., 'companybd.com').
    """
    if website_link:
        search_query=f"{company_name}OR{website_link} official site Bngladesh"
    else:
        search_query=f"{company_name} official site Bangladesh"
    try:
        search_results=TavilySearchResults.search(search_query,max_results=3)
        summary = f"Search Findings for query [{search_query}]:\n"
        for idx,result in enumerate(search_results.results):
            tile=result.get("title","No title")
            url=result.get("url","No URL")
            snippet=result.get("content","No content")
            summary+=f"Result {idx+1}:\nTitle: {tile}\nURL: {url}\nSnippet: {snippet}\n\n"
        return summary
    except Exception as e:
        return f"Search failed for '{search_query}'. Error details: {str(e)}"


In [ ]:
def input_normalizer_node(state:FraudState):
    
    email_pattern=r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    raw_email=list(re.findall(email_pattern,state["job_text"]))
    free_email_domains=["gmail.com","yahoo.com","hotmail.com","outlook.com","aol.com"]
    for email in raw_email:
        domain=email.split("@")[-1].lower()
        if domain in free_email_domains:
            state["is_email_free"]=True
        else:
            state["is_email_free"]=False

    website_pattern=r'https?://(?:www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b(?:[-a-zA-Z0-9()@:%_\+.~#?&//=]*)'
    raw_website=list(re.findall(website_pattern,state["job_text"]))  
    if len(raw_website)>0:
        website_available=True
    else:
        website_available=False
              
    job_text=state["job_text"].strip()
    return{
        "job_text":job_text,
        "emailaddress":raw_email,
        "is_email_free":state["is_email_free"],
        "website_link":raw_website,
        "website_available":website_available,
        "evidance_log":[f"Step 1: Extracted emails: {raw_email}, Free email: {state['is_email_free']}, Extracted websites: {raw_website}, Website available: {website_available}"]
    }

NameError: name 'FraudState' is not defined

In [ ]:
def key_word_extractor(state:FraudState):
    """This node extracts suspicious phrases from the job text using the structured LLM."""
    job_text=state["job_text"]
    response:SuspiciousPhraseExtractor=structured_llm(f"You are a recruitment fraud analyst in Bangladesh. Extract any phrases "
        "from this job circular that request money upfront (e.g., bKash, registration fee), "
        "offer unrealistic pay for basic skills, or pressure the applicant to act quickly:\n\n"
        f"{job_text}\n\n" "if there are no such phrases,return an empty list.")
    extracted_phrases=response.phrases
    return{
        "suspicious_keyword":extracted_phrases,
        "evidance_log":[f"Step:2 Extracted suspicious phrases: {extracted_phrases}"]
    }
    